# 🧠 agents_explore — the two graphs and the one place judgment lives

The whole system has been deterministic until now — contracts, controller, provisioner
all replay identically. This notebook is where **judgment** enters, in exactly two slots
(hard rule 1): the consumer's *accept/reject* and the provider's *quote/decline*.
Everything else in both agent graphs is mechanical plumbing.

This tour uses **stub LLMs and stub tools** so it runs headless anywhere (the real
models are slow on a shared CPU box). The structure you see is exactly what the live
agents run; only the two judgment slots swap a fake for a real `LLMClient`.

**Companions:** [`docs/06-agents-spec.md`](../../docs/06-agents-spec.md) ·
`agents/src/agents/{llm,decision,consumer_graph,provider_graph}.py` · story ch. 9.

## 1. The LLM client's real job: make the backend irrelevant (ADR-001)

`LLMClient.structured()` asks any OpenAI-compatible endpoint for JSON, then **validates
it against a pydantic model and retries** until it parses or the budget runs out. The
caller gets a guaranteed-valid object or a clean error — never a raw string. That
guarantee is what lets the same code run on Ollama, vLLM, or a stub. Here's the loop
with a scripted backend (first two replies are junk, the third is valid):

In [1]:
from pydantic import BaseModel
from agents.llm import LLMClient, LLMConfig

class Decision(BaseModel):
    accept: bool
    reason: str

class ScriptedChat:
    def __init__(self, replies): self.replies, self.calls = replies, 0
    def create(self, **k):
        r = self.replies[self.calls]; self.calls += 1
        return type("R", (), {"choices": [type("C", (), {"message": type("M", (), {"content": r})})]})

client = LLMClient(LLMConfig(base_url="x", model="m", api_key="k"))
chat = ScriptedChat(["not json", '{"accept": false}', '{"accept": true, "reason": "fits budget"}'])
client._client = type("O", (), {"chat": type("Ch", (), {"completions": chat})})()

out = client.structured("system", "user", Decision)
print(f"after {chat.calls} attempts → {out}   (a guaranteed-valid Decision)")


after 3 attempts → accept=True reason='fits budget'   (a guaranteed-valid Decision)


## 2. The consumer graph: discover → quote → **decide** → settle → activate → report

A LangGraph state machine. Every node is mechanical except `decide`, the LLM slot. An
`accept` runs the full purchase; a `decline` exits gracefully having bought nothing.
Watch both branches with a stubbed decision:

In [2]:
from a2a_interfaces import DecisionOutput
from a2a_interfaces.fixtures import BANDWIDTH_NEED, CANONICAL_SIGNED_OFFER
from agents.consumer_graph import ConsumerState, build_consumer_graph

class FakeLLM:
    def __init__(self, accept): self.d = DecisionOutput(accept=accept, reason="scripted")
    def structured(self, *a): return self.d

class StubTools:
    def quote(self, need): return CANONICAL_SIGNED_OFFER
    def settle(self, offer): return 7
    def activate(self, eid): return f"ent{eid}-a0"

for accept in (True, False):
    result = build_consumer_graph(FakeLLM(accept), StubTools()).invoke(
        ConsumerState(need=BANDWIDTH_NEED, budget_tok=15))
    print(f"\n--- consumer decides accept={accept} ---")
    for line in result["transcript"]:
        print("  " + line)



--- consumer decides accept=True ---
  quote: 10 TOK for bandwidth
  decide: accept — scripted
  settle: minted entitlement #7
  activate: session ent7-a0 active
  report: service running

--- consumer decides accept=False ---
  quote: 10 TOK for bandwidth
  decide: decline — scripted
  exit: offer declined, nothing purchased


## 3. The provider graph: two kinds of "no"

The provider can decline for two reasons, and they are different in kind:
- **admission control** — *deterministic* arithmetic over a capacity ledger. "Can I
  physically commit this without overselling?" is not a judgment (story ch. 8).
- **the quote** — the provider's *LLM* slot: given capacity exists, price it or decline.

Here is ch. 8's no-overselling as three lines: 60 Mbps twice against a 100 Mbps ledger.

In [3]:
from a2a_interfaces import SignedOffer, Decline
from a2a_interfaces.fixtures import WINDOW
from a2a_interfaces import BandwidthNeed
from agents.provider_graph import CapacityLedger, ProviderState, QuoteDecision, build_provider_graph

class QuoteLLM:
    def structured(self, *a): return QuoteDecision(quote=True, price_tok=10, reason="fair")
class SignTool:
    def sign_offer(self, need, price): return CANONICAL_SIGNED_OFFER

def need(bps): return BandwidthNeed(src="hostA", dst="hostB", capacity_bps=bps, qos_class=1, window=WINDOW)

ledger = CapacityLedger(capacity_bps=100_000_000)
graph = build_provider_graph(QuoteLLM(), SignTool(), ledger)

for i in (1, 2):
    r = graph.invoke(ProviderState(need=need(60_000_000)))
    kind = "SignedOffer ✓" if isinstance(r["result"], SignedOffer) else f"Decline ✗ ({r['result'].reason})"
    print(f"request {i}: 60 Mbps → {kind}   [ledger now {ledger.available((WINDOW.start, WINDOW.end))//1_000_000} Mbps free]")


request 1: 60 Mbps → SignedOffer ✓   [ledger now 40 Mbps free]
request 2: 60 Mbps → Decline ✗ (insufficient capacity in the requested window)   [ledger now 40 Mbps free]


## 4. Where judgment lives — the whole map

| Slot | Who | Deterministic? | File |
|---|---|---|---|
| accept/reject an offer | consumer | **no — LLM** | `decision.py` |
| quote/decline (pricing) | provider | **no — LLM** | `provider_graph.py` quote node |
| admission (overselling) | provider | yes — ledger arithmetic | `provider_graph.py` admit node |
| authorize activation | controller | yes — predicate | `controller/domain.py` |
| everything else | — | yes | — |

Exactly two LLM slots (rule 1). Everything the *network* does is deterministic; the LLM
only decides whether a deal happens and at what price.

## Experiments to try

- Point the consumer at a real model: `from agents.llm import LLMClient` with
  `LLMConfig.from_env()` (needs a running Ollama — slow on a shared box, but real).
- Make the provider's `QuoteLLM` return `quote=False` — watch the reserved capacity get
  released (a business decline must not consume the slot).
- Request 60 then 50 Mbps against the 100 ledger — does the second fit? Predict first.
- Trace `build_consumer_graph`'s conditional edge: which one line turns a `decline` into
  a graceful exit instead of a settlement?